# ДЗ 7. CNN, часть 2: transfer learning и сегментация

`Imagenette_improved.ipynb` с лекции обучал ResNet18 **с нуля** (`weights=None`) с аугментациями и early stopping. Здесь — transfer learning и переход от классификации к **сегментации** — определению класса для каждого пикселя, а не всего изображения целиком.

Для transfer learning используем EuroSAT из ДЗ 6. Для сегментации — **Oxford-IIIT Pet** с трёхклассовой trimap-разметкой. Бинарная постановка: питомец против фона.

Полный прогон рассчитан на GPU Colab; при нехватке времени можно уменьшить вычислительный бюджет, сохранив логику экспериментов.


In [ ]:
import torch
import torch.nn as nn
import torchvision
import torchvision.models as models
from torchvision import transforms
import torchvision.transforms.functional as TF
from torch.utils.data import DataLoader, Subset, Dataset
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import random, time, gc
from IPython.display import display

torch.manual_seed(42); random.seed(42); np.random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
NUM_WORKERS = 2 if device.type == 'cuda' else 0
BATCH_SIZE = 64 if device.type == 'cuda' else 32
print('Device:',device)
if device.type=='cuda': print('GPU:',torch.cuda.get_device_name(0))

weights = models.ResNet18_Weights.DEFAULT
plain_transform = transforms.Compose([
    transforms.Resize((224,224)), transforms.ToTensor(),
    transforms.Normalize((0.485,0.456,0.406),(0.229,0.224,0.225))
])
train_transform = transforms.Compose([
    transforms.Resize((224,224)), transforms.RandomHorizontalFlip(), transforms.ToTensor(),
    transforms.Normalize((0.485,0.456,0.406),(0.229,0.224,0.225))
])

base = torchvision.datasets.EuroSAT(root='/content/case3_data',transform=None,download=True)
labels=np.array(base.targets); classes=base.classes
rng=np.random.default_rng(42)
train_idx,val_idx,test_idx=[],[],[]
for c in np.unique(labels):
    idx=np.where(labels==c)[0]; rng.shuffle(idx); n=len(idx); a=int(.70*n); b=int(.85*n)
    train_idx.extend(idx[:a]); val_idx.extend(idx[a:b]); test_idx.extend(idx[b:])
train_idx,val_idx,test_idx=map(np.array,(train_idx,val_idx,test_idx))

train_base=torchvision.datasets.EuroSAT(root='/content/case3_data',transform=train_transform,download=False)
eval_base=torchvision.datasets.EuroSAT(root='/content/case3_data',transform=plain_transform,download=False)
train_ds=Subset(train_base,train_idx); val_ds=Subset(eval_base,val_idx); test_ds=Subset(eval_base,test_idx)

def loader(ds,shuffle=False,seed=42):
    return DataLoader(ds,batch_size=BATCH_SIZE,shuffle=shuffle,num_workers=NUM_WORKERS,
                      pin_memory=device.type=='cuda',generator=torch.Generator().manual_seed(seed))
train_loader,val_loader,test_loader=loader(train_ds,True),loader(val_ds),loader(test_ds)

def train_cls_epoch(model,dl,opt):
    model.train(); total=correct=0; loss_sum=0
    for x,y in dl:
        x,y=x.to(device),y.to(device); opt.zero_grad(set_to_none=True)
        out=model(x); loss=nn.functional.cross_entropy(out,y)
        loss.backward(); opt.step()
        bs=y.size(0); total+=bs; loss_sum+=loss.item()*bs; correct+=(out.argmax(1)==y).sum().item()
    return loss_sum/total,correct/total

@torch.no_grad()
def eval_cls(model,dl):
    model.eval(); total=correct=0; loss_sum=0
    for x,y in dl:
        x,y=x.to(device),y.to(device); out=model(x); loss=nn.functional.cross_entropy(out,y); bs=y.size(0)
        total+=bs; loss_sum+=loss.item()*bs; correct+=(out.argmax(1)==y).sum().item()
    return loss_sum/total,correct/total

def make_resnet(pretrained=True):
    m=models.resnet18(weights=weights if pretrained else None)
    m.fc=nn.Linear(m.fc.in_features,len(classes))
    return m

def fit_resnet(model,epochs=8,strategy='full'):
    model.to(device)
    opt=torch.optim.Adam(filter(lambda p:p.requires_grad,model.parameters()),lr=1e-3)
    hist={'val_acc':[],'epoch_time':[]}
    for ep in range(epochs):
        if strategy=='classifier_only':
            model.eval(); model.fc.train()
        elif strategy=='last_block':
            model.eval(); model.layer4.train(); model.fc.train()
        else:
            model.train()
        t=time.perf_counter(); train_cls_epoch(model,train_loader,opt); _,va=eval_cls(model,val_loader)
        hist['val_acc'].append(va); hist['epoch_time'].append(time.perf_counter()-t)
        print(f'{ep+1:02d}/{epochs}: val={va:.4f}')
    return model,hist

def cleanup(*objs):
    for obj in objs: del obj
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

def epoch_80(history):
    target=.8*history['val_acc'][-1]
    return next((i for i,v in enumerate(history['val_acc'],1) if v>=target),None)


## Уровень 1. Базовый

### Задача 1.1. Transfer learning против обучения с нуля

In [ ]:
# 1.1 ResNet18: scratch vs pretrained
scratch,h_s=fit_resnet(make_resnet(False),epochs=8,strategy='full')
acc_s=eval_cls(scratch,test_loader)[1]
cleanup(scratch)
pretrained,h_p=fit_resnet(make_resnet(True),epochs=8,strategy='full')
acc_p=eval_cls(pretrained,test_loader)[1]
print(f'Test accuracy: scratch={acc_s:.4f}, pretrained={acc_p:.4f}')
print(f'Эпоха достижения 80% от финальной val accuracy: scratch={epoch_80(h_s)}, pretrained={epoch_80(h_p)}')
plt.figure(figsize=(9,4)); plt.plot(h_s['val_acc'],label='scratch'); plt.plot(h_p['val_acc'],label='pretrained')
plt.xlabel('Epoch'); plt.ylabel('Val accuracy'); plt.title('ResNet18: scratch vs pretrained'); plt.legend(); plt.grid(alpha=.2); plt.show()


### Задача 1.2. Стратегии дообучения: заморозка слоёв

In [ ]:
# 1.2 Стратегии заморозки
rows=[]
for strategy in ['classifier_only','last_block','full']:
    print(f'
[1.2 {len(rows)+1}/3] {strategy}')
    torch.manual_seed(42); np.random.seed(42); random.seed(42)
    m=make_resnet(True)
    if strategy=='classifier_only':
        for p in m.parameters(): p.requires_grad=False
        for p in m.fc.parameters(): p.requires_grad=True
    elif strategy=='last_block':
        for p in m.parameters(): p.requires_grad=False
        for p in m.layer4.parameters(): p.requires_grad=True
        for p in m.fc.parameters(): p.requires_grad=True
    else:
        for p in m.parameters(): p.requires_grad=True
    trainable=sum(p.numel() for p in m.parameters() if p.requires_grad)
    t=time.perf_counter(); m,h=fit_resnet(m,epochs=8,strategy=strategy); elapsed=time.perf_counter()-t
    rows.append({'strategy':strategy,'trainable_params':trainable,'epoch_time':elapsed/8,'test_accuracy':eval_cls(m,test_loader)[1]})
    cleanup(m)
freeze_results=pd.DataFrame(rows); display(freeze_results)


## Уровень 2. Средний

### Задача 2.1. U-Net с нуля для бинарной сегментации

In [ ]:
# 2.1 U-Net + Oxford-IIIT Pet
class PetSeg(Dataset):
    def __init__(self,base,idx,train=False,size=128):
        self.base,self.idx,self.train,self.size=base,idx,train,size
    def __len__(self): return len(self.idx)
    def __getitem__(self,i):
        image,mask=self.base[int(self.idx[i])]
        image=TF.resize(image,[self.size,self.size],antialias=True)
        mask=TF.resize(mask,[self.size,self.size],interpolation=TF.InterpolationMode.NEAREST)
        if self.train:
            if torch.rand(())<.5: image=TF.hflip(image); mask=TF.hflip(mask)
            angle=float(torch.empty(1).uniform_(-15,15))
            image=TF.rotate(image,angle,interpolation=TF.InterpolationMode.BILINEAR)
            mask=TF.rotate(mask,angle,interpolation=TF.InterpolationMode.NEAREST)
        image=TF.to_tensor(image); image=TF.normalize(image,(.485,.456,.406),(.229,.224,.225))
        mask=np.asarray(mask,dtype=np.int64); mask=np.isin(mask,[1,3]).astype('float32')
        return image,torch.from_numpy(mask).unsqueeze(0)

pet_base=torchvision.datasets.OxfordIIITPet(
    root='/content/case3_data',split='trainval',target_types='segmentation',download=True
)
idx=np.arange(len(pet_base)); rng=np.random.default_rng(42); rng.shuffle(idx)
idx=idx[:min(1500,len(idx))]
n=len(idx); a=int(.70*n); b=int(.85*n)
pet_train,pet_val,pet_test_idx=idx[:a],idx[a:b],idx[b:]
pet_train_ds=PetSeg(pet_base,pet_train,True)
pet_val_ds=PetSeg(pet_base,pet_val)
pet_test_ds=PetSeg(pet_base,pet_test_idx)
pet_train_loader=DataLoader(pet_train_ds,batch_size=16,shuffle=True,num_workers=NUM_WORKERS,pin_memory=device.type=='cuda')
pet_val_loader=DataLoader(pet_val_ds,batch_size=16,shuffle=False,num_workers=NUM_WORKERS,pin_memory=device.type=='cuda')
pet_test_loader=DataLoader(pet_test_ds,batch_size=16,shuffle=False,num_workers=NUM_WORKERS,pin_memory=device.type=='cuda')
print(f'Pet split: train={len(pet_train_ds)} val={len(pet_val_ds)} test={len(pet_test_ds)}')
print('Mask unique values:',torch.unique(pet_train_ds[0][1]).tolist())

class DoubleConv(nn.Module):
    def __init__(self,a,b):
        super().__init__()
        self.net=nn.Sequential(
            nn.Conv2d(a,b,3,padding=1,bias=False),nn.BatchNorm2d(b),nn.ReLU(inplace=True),
            nn.Conv2d(b,b,3,padding=1,bias=False),nn.BatchNorm2d(b),nn.ReLU(inplace=True)
        )
    def forward(self,x): return self.net(x)

class UNet(nn.Module):
    def __init__(self,base=32):
        super().__init__()
        self.pool=nn.MaxPool2d(2)
        self.e1=DoubleConv(3,base); self.e2=DoubleConv(base,base*2)
        self.e3=DoubleConv(base*2,base*4); self.e4=DoubleConv(base*4,base*8)
        self.b=DoubleConv(base*8,base*16)
        self.u4=nn.ConvTranspose2d(base*16,base*8,2,2); self.d4=DoubleConv(base*16,base*8)
        self.u3=nn.ConvTranspose2d(base*8,base*4,2,2); self.d3=DoubleConv(base*8,base*4)
        self.u2=nn.ConvTranspose2d(base*4,base*2,2,2); self.d2=DoubleConv(base*4,base*2)
        self.u1=nn.ConvTranspose2d(base*2,base,2,2); self.d1=DoubleConv(base*2,base)
        self.out=nn.Conv2d(base,1,1)
    def forward(self,x):
        e1=self.e1(x); e2=self.e2(self.pool(e1)); e3=self.e3(self.pool(e2)); e4=self.e4(self.pool(e3)); b=self.b(self.pool(e4))
        d4=self.d4(torch.cat([self.u4(b),e4],1)); d3=self.d3(torch.cat([self.u3(d4),e3],1))
        d2=self.d2(torch.cat([self.u2(d3),e2],1)); d1=self.d1(torch.cat([self.u1(d2),e1],1))
        return self.out(d1)

def iou(logits,target):
    pred=(torch.sigmoid(logits)>=.5).float()
    inter=(pred*target).sum((1,2,3)); union=((pred+target)>0).float().sum((1,2,3))
    return torch.where(union>0,inter/union,torch.ones_like(union)).mean().item()

def seg_epoch(model,dl,opt=None):
    model.train(opt is not None); loss_fn=nn.BCEWithLogitsLoss(); ls=ios=n=0
    for x,y in dl:
        x,y=x.to(device),y.to(device)
        if opt: opt.zero_grad(set_to_none=True)
        out=model(x); loss=loss_fn(out,y)
        if opt: loss.backward(); opt.step()
        bs=x.size(0); n+=bs; ls+=loss.item()*bs; ios+=iou(out.detach(),y)*bs
    return ls/n,ios/n

def naive_iou(dl,fill):
    vals=[]
    for _,y in dl:
        pred=torch.full_like(y,float(fill)); inter=(pred*y).sum((1,2,3)); union=((pred+y)>0).float().sum((1,2,3))
        vals.extend(torch.where(union>0,inter/union,torch.ones_like(union)).tolist())
    return float(np.mean(vals))

baseline_bg,baseline_fg=naive_iou(pet_test_loader,0),naive_iou(pet_test_loader,1)
model=UNet().to(device); opt=torch.optim.Adam(model.parameters(),lr=1e-3); best=-1; best_state=None
hist={'train_iou':[],'val_iou':[]}
for ep in range(15):
    trl,tri=seg_epoch(model,pet_train_loader,opt); vl,vi=seg_epoch(model,pet_val_loader)
    hist['train_iou'].append(tri); hist['val_iou'].append(vi)
    if vi>best: best=vi; best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
    print(f'{ep+1:02d}/15: train IoU={tri:.4f} val IoU={vi:.4f}')
model.load_state_dict(best_state); _,test_iou=seg_epoch(model,pet_test_loader)
print(f'Baseline all-background IoU: {baseline_bg:.4f}')
print(f'Baseline all-foreground IoU: {baseline_fg:.4f}')
print(f'U-Net test IoU: {test_iou:.4f}')

fig,ax=plt.subplots(figsize=(8,4))
ax.plot(hist['train_iou'],label='train'); ax.plot(hist['val_iou'],label='val')
ax.set(xlabel='Epoch',ylabel='IoU',title='U-Net training'); ax.legend(); ax.grid(alpha=.2)
plt.show()

@torch.no_grad()
def show_masks(model,ds,n=4):
    model.eval(); fig,ax=plt.subplots(n,3,figsize=(9,3*n)); ax=np.atleast_2d(ax)
    for r in range(n):
        x,y=ds[r]; p=(torch.sigmoid(model(x.unsqueeze(0).to(device)))[0,0].cpu()>=.5)
        vis=x*torch.tensor((.229,.224,.225)).view(3,1,1)+torch.tensor((.485,.456,.406)).view(3,1,1)
        ax[r,0].imshow(vis.clamp(0,1).permute(1,2,0)); ax[r,1].imshow(y[0],cmap='gray'); ax[r,2].imshow(p,cmap='gray')
        ax[r,0].set_title('Image'); ax[r,1].set_title('Ground truth'); ax[r,2].set_title('Prediction')
        for a in ax[r]: a.axis('off')
    plt.tight_layout(); plt.show()
show_masks(model,pet_test_ds)


---
Численные результаты формируются непосредственно при запуске ноутбука.